# 04 SVM Aspect Classification

Executable Phase 9A notebook for preparing the candidate SVM aspect-classification dataset. This notebook prepares and inspects the dataset only; it does not fit TF-IDF, train SVM, or create model artifacts.


## CRISP-DM Stage

Modeling preparation. The output of this notebook is a filtered weak-label dataset that can be used in Phase 9B for TF-IDF and SVM training experiments.


## SVM Role in SentiRank

SVM is used for aspect classification, not sentiment classification. Sentiment classification is handled by the final IndoBERT candidate (`run_3_weighted_loss_lr_1e-5`). The SVM model will later classify actionable review aspects so that validated criteria can support AHP and Fuzzy AHP analysis.


## Weak Label Limitation

The input aspect labels are weak labels from keyword-based rules. They are not expert-validated ground truth. Phase 9A reduces label noise by excluding `General` fallback rows and low-confidence labels before any SVM training is attempted.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "ml-service").exists():
            return candidate
    raise RuntimeError("Could not find SentiRank project root from current working directory.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
DATASETS_DIR = PROJECT_ROOT / "datasets"
EDA04_DIR = DATASETS_DIR / "outputs" / "eda" / "04_svm"
FIG04_DIR = PROJECT_ROOT / "docs" / "figures" / "04_svm"
INPUT_DATASET = DATASETS_DIR / "processed" / "reviews_with_aspect_labels_refined.csv"
OUTPUT_DATASET = DATASETS_DIR / "processed" / "svm" / "svm_aspect_dataset.csv"
SUMMARY_PATH = EDA04_DIR / "svm_aspect_dataset_summary.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_DATASET}")
print(f"Output dataset: {OUTPUT_DATASET}")


## Dataset Preparation Strategy

Phase 9A keeps only actionable and sufficiently reliable weak labels:

- Text column: `text_svm`
- Target label: `aspect_label`
- Excluded labels: `General`
- Allowed confidence values: `medium`, `high`
- Candidate target labels: `Performance & Stability`, `Ads Experience`, `Subscription & Pricing`, `Features & Content`, `Audio Quality`, `UI/UX`, `Account/Login`

`General` is a fallback label only and must not be used as an SVM target class for the final experiment or as an AHP/Fuzzy AHP criterion.


## Run Dataset Preparation Script

This script filters the refined weak-label dataset and generates small aggregate metrics and figures. It does not train SVM, fit TF-IDF, create vectorizers, or create model artifacts.


In [ ]:
RUN_DATASET_PREPARATION = True

if RUN_DATASET_PREPARATION:
    command = [sys.executable, "scripts/prepare_svm_aspect_dataset.py"]
    result = subprocess.run(command, cwd=ML_SERVICE_DIR, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Dataset preparation failed with return code {result.returncode}")
else:
    display(Markdown("Dataset preparation skipped. Set `RUN_DATASET_PREPARATION = True` to regenerate Phase 9A outputs."))


## Load Summary JSON

The summary explains how many rows were removed by each Phase 9A filter and records the weak-label methodology note.


In [ ]:
def load_json(path: Path):
    if not path.exists():
        display(Markdown(f"Missing JSON: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def show_json_as_table(payload, title: str):
    if payload is None:
        return
    display(Markdown(f"### {title}"))
    table = pd.json_normalize(payload, sep=".").T.reset_index()
    table.columns = ["field", "value"]
    display(table)


summary = load_json(SUMMARY_PATH)
show_json_as_table(summary, "SVM Aspect Dataset Summary")


## Distribution Tables

These tables are generated under `datasets/outputs/eda/04_svm/` for thesis reporting and future dashboard use.


In [ ]:
def load_csv(path: Path):
    if not path.exists():
        display(Markdown(f"Missing CSV: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    return pd.read_csv(path)


metric_tables = {
    "Aspect label distribution": EDA04_DIR / "svm_aspect_label_distribution.csv",
    "Aspect confidence distribution": EDA04_DIR / "svm_aspect_confidence_distribution.csv",
    "Aspect by sentiment distribution": EDA04_DIR / "svm_aspect_by_sentiment_distribution.csv",
}

loaded_tables = {}
for title, path in metric_tables.items():
    data = load_csv(path)
    if data is not None:
        loaded_tables[title] = data
        display(Markdown(f"### {title}"))
        display(data)


## Figure Outputs

Figures are generated under `docs/figures/04_svm/` using matplotlib only.


In [ ]:
def show_figures(paths: list[Path]) -> None:
    for path in paths:
        if path.exists():
            display(Markdown(f"**{path.relative_to(PROJECT_ROOT)}**"))
            display(Image(filename=str(path)))
        else:
            display(Markdown(f"Missing figure: `{path.relative_to(PROJECT_ROOT)}`"))


show_figures([
    FIG04_DIR / "svm_aspect_label_distribution.png",
    FIG04_DIR / "svm_aspect_confidence_distribution.png",
    FIG04_DIR / "svm_aspect_by_sentiment_distribution.png",
])


## Interpretation Notes

The cell below derives the key Phase 9A interpretation from the generated summary.


In [ ]:
notes = []
if summary:
    notes.append(f"Input rows: `{summary.get('total_input_rows'):,}`.")
    notes.append(f"Rows after excluding `General`: `{summary.get('rows_after_excluding_general'):,}`.")
    notes.append(f"Rows after confidence filter: `{summary.get('rows_after_confidence_filter'):,}`.")
    notes.append(f"Final SVM candidate dataset rows: `{summary.get('final_dataset_rows'):,}`.")
    notes.append(f"Aspect distribution: `{summary.get('aspect_distribution')}`.")
    notes.append(f"Confidence distribution: `{summary.get('confidence_distribution')}`.")
    notes.append(summary.get("methodology_note", ""))
else:
    notes.append("Summary is missing. Run the dataset preparation cell first.")

notes.append("The low-count classes, especially `UI/UX`, should be handled in Phase 9B through explicit training/evaluation strategy rather than by silently changing Phase 9A filters.")
notes.append("Next step: Phase 9B SVM TF-IDF and training.")

display(Markdown("\n".join(f"- {note}" for note in notes if note)))


## Expected Outputs

- Ignored local dataset: `datasets/processed/svm/svm_aspect_dataset.csv`
- Metrics: `datasets/outputs/eda/04_svm/`
- Figures: `docs/figures/04_svm/`

Do not commit generated processed CSV datasets. Commit only the reproducible script, notebook, small EDA metrics, and figures.
